# Hướng Dẫn PyTorch Từ Cơ Bản Đến Nâng Cao

Notebook này đi kèm phần lý thuyết, ví dụ minh họa và bài tập thực hành. Chạy từng ô theo thứ tự để hiểu rõ hơn.

## 1. PyTorch là gì?

PyTorch là một thư viện deep learning viết bằng Python, mặc định chạy trên CPU nhưng có thể tăng tốc bằng GPU thông qua CUDA. PyTorch dùng cách tiếp cận "define-by-run" — đồ thị tính toán được xây dựng động ngay trong lúc chạy code, giúp debug và tùy biến dễ dàng hơn.

Ba đặc điểm nổi bật:
- Đồ thị tính toán động (dynamic computation graph)
- Tự động tính đạo hàm (autograd)
- Hỗ trợ tăng tốc GPU qua CUDA

Cài đặt (nếu chưa có):
```
pip install torch torchvision torchaudio
```


In [ ]:
import torch
import torch.nn as nn

print(torch.__version__)

## 2. Tensor trong PyTorch

Tensor là cấu trúc dữ liệu nền tảng của PyTorch, tương tự mảng NumPy nhưng hỗ trợ GPU và tính đạo hàm tự động.

In [ ]:
# Tensor 1 chiều
x = torch.tensor([1.0, 2.0, 3.0])
print("Tensor 1D:", x)

# Tensor 2 chiều toàn số 0
y = torch.zeros((3, 3))
print("Tensor 2D:", y)

### Các phép toán cơ bản trên tensor

In [ ]:
a = torch.tensor([1.0, 2.0])
b = torch.tensor([3.0, 4.0])

print("Cộng theo từng phần tử:", a + b)
print("Nhân ma trận:", torch.matmul(a.view(2, 1), b.view(1, 2)))

### Reshape và Transpose

`reshape()` và `view()` đều thay đổi hình dạng tensor mà không đổi dữ liệu. `view()` yêu cầu tensor liên tục trong bộ nhớ, còn `reshape()` linh hoạt hơn (tự tạo bản sao khi cần).

In [ ]:
t = torch.tensor([[1, 2, 3, 4],
                   [5, 6, 7, 8],
                   [9, 10, 11, 12]])

print("Reshape thành (6, 2):")
print(t.reshape(6, 2))

print("Transpose (đổi chiều hàng-cột):")
print(t.transpose(0, 1))

## 3. Autograd và đồ thị tính toán

Module `autograd` tự động tính gradient phục vụ cho lan truyền ngược (backpropagation).

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()
print(x.grad)   # kỳ vọng: tensor(4.)

Giải thích:
- `y = x ** 2` được ghi lại vào đồ thị tính toán.
- `y.backward()` tính đạo hàm của `y` theo `x`.
- Vì `y = x²` nên `dy/dx = 2x`; với `x = 2`, gradient = 4.

## 4. Xây dựng mạng neural với `torch.nn`

Một mạng neural được xây dựng bằng cách kế thừa `torch.nn.Module`:
- `nn.Linear(in_features, out_features)`: lớp fully-connected.
- Hàm kích hoạt: `torch.relu`, `torch.sigmoid`, `torch.softmax`.
- `forward()`: quy định luồng dữ liệu đi qua mạng.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x

model = NeuralNetwork()
print(model)

### Loss function và Optimizer

- Hàm mất mát đo sai số giữa dự đoán và nhãn thật.
- Optimizer cập nhật trọng số dựa trên gradient.

Ví dụ: `nn.BCELoss()` (binary cross-entropy) + `optim.Adam()`.

In [ ]:
model = NeuralNetwork()
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

### Vòng lặp huấn luyện

5 bước cố định mỗi epoch:
1. `optimizer.zero_grad()` — xóa gradient cũ
2. Forward pass — `model(inputs)`
3. Tính loss — `criterion(outputs, targets)`
4. `loss.backward()` — lan truyền ngược
5. `optimizer.step()` — cập nhật trọng số

In [ ]:
inputs = torch.randn((100, 10))
targets = torch.randint(0, 2, (100, 1)).float()
epochs = 20

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

## 5. PyTorch so với TensorFlow

| Tiêu chí | PyTorch | TensorFlow |
|---|---|---|
| Đồ thị tính toán | Động | Động theo mặc định (eager execution) |
| Độ dễ sử dụng | Rất Pythonic, dễ debug | Đường học hơi dốc hơn |
| Hiệu năng | Nhanh nhờ eager execution | Tối ưu cho triển khai quy mô lớn |
| Triển khai | TorchScript & ONNX | TensorFlow Serving & TF Lite |
| Phổ biến | Rất phổ biến trong nghiên cứu | Phổ biến, thiên về production |

## 6. Ứng dụng thực tế

- **Thị giác máy tính**: phân loại ảnh, phát hiện vật thể, phân đoạn ảnh (CNN, Transformer/ViT)
- **NLP**: Transformer, RNN, LSTM cho sinh văn bản, phân tích cảm xúc
- **Học tăng cường**: DQN, Policy Gradient, Actor-Critic

## 7. Bài tập thực hành cơ bản

Hoàn thành các ô code bên dưới (đã để sẵn khung `# TODO`).

**Bài 1:** Tạo tensor 1D gồm 5 số bất kỳ, và tensor 2D (4,4) toàn số 1. In dtype và shape của cả hai.

In [ ]:
# TODO: Bài 1


**Bài 2:** Cho `a = [2, 4, 6]`, `b = [1, 3, 5]`. Tính cộng, trừ, nhân từng phần tử, và tích vô hướng (dot product).

In [ ]:
# TODO: Bài 2


**Bài 3:** Tạo tensor (2,6) chứa số 1→12. Dùng `reshape()` về (3,4), rồi `transpose()` thành (4,3). In từng bước.

In [ ]:
# TODO: Bài 3


**Bài 4:** Cho `x = torch.tensor(3.0, requires_grad=True)`, `y = 2*x**3 + 5*x`. Gọi `y.backward()`, in `x.grad` và giải thích (gợi ý: đạo hàm là `6x² + 5`).

In [ ]:
# TODO: Bài 4


**Bài 5:** Viết mạng neural 2 lớp ẩn, đầu vào 4 đặc trưng, đầu ra 3 lớp (dùng `softmax` ở lớp cuối).

In [ ]:
# TODO: Bài 5


**Bài 6:** Viết vòng lặp huấn luyện đầy đủ (data giả, loss, optimizer) cho mạng ở Bài 5, chạy 10 epoch, in loss mỗi 2 epoch.

In [ ]:
# TODO: Bài 6


## 8. Lý thuyết nâng cao

### 8.1. Dataset và DataLoader

`torch.utils.data.Dataset` đóng gói dữ liệu; `DataLoader` chia thành batch, tự shuffle và load song song.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = MyDataset(inputs, targets)
loader = DataLoader(dataset, batch_size=16, shuffle=True)
print(len(loader), "batches")

### 8.2. Chạy trên GPU (CUDA)

Chuyển cả model lẫn dữ liệu sang cùng device.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Đang dùng:", device)
model = model.to(device)

### 8.3. Mạng tích chập (CNN)

Với dữ liệu ảnh, dùng thêm `nn.Conv2d`, `nn.MaxPool2d`.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(16 * 16 * 16, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)  # flatten
        x = self.fc(x)
        return x

cnn = SimpleCNN()
print(cnn)

### 8.4. Learning rate scheduler

Giảm dần learning rate theo epoch để hội tụ ổn định hơn.

In [ ]:
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
# Gọi scheduler.step() sau mỗi epoch trong vòng lặp huấn luyện

### 8.5. Regularization: Dropout và BatchNorm

- `nn.Dropout(p=0.5)`: ngẫu nhiên tắt một số neuron khi huấn luyện, chống overfitting.
- `nn.BatchNorm1d` / `nn.BatchNorm2d`: chuẩn hóa đầu ra mỗi lớp, giúp huấn luyện nhanh và ổn định hơn.

In [ ]:
class RegularizedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 32)
        self.bn1 = nn.BatchNorm1d(32)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = torch.sigmoid(self.fc2(x))
        return x

### 8.6. Lưu và load mô hình

In [ ]:
torch.save(model.state_dict(), "model.pth")

model2 = NeuralNetwork()
model2.load_state_dict(torch.load("model.pth"))
model2.eval()

### 8.7. Chế độ `train()` và `eval()`

`model.train()` bật dropout/batchnorm ở chế độ huấn luyện; `model.eval()` chuyển sang suy luận. Luôn dùng `torch.no_grad()` khi dự đoán để tiết kiệm bộ nhớ.

In [ ]:
model.eval()
with torch.no_grad():
    test_inputs = torch.randn((5, 10))
    predictions = model(test_inputs)
    print(predictions)

## 9. Bài tập nâng cao

Hoàn thành các ô code bên dưới.

**Bài 1 (nâng cao):** Viết class `Dataset` tùy chỉnh cho mảng NumPy features/labels bất kỳ, kết hợp `DataLoader` (batch_size=32, shuffle=True). Lặp qua loader và in shape mỗi batch.

In [ ]:
# TODO: Bài 1 nâng cao


**Bài 2 (nâng cao):** Viết lại vòng lặp huấn luyện Bài 6 (cơ bản) dùng `DataLoader` thay vì đưa toàn bộ dữ liệu 1 lần, đồng thời tự động chọn `cuda` nếu có GPU.

In [ ]:
# TODO: Bài 2 nâng cao


**Bài 3 (nâng cao):** Tạo ảnh giả (batch, 3, 32, 32) bằng `torch.randn`, đưa qua `SimpleCNN` với 2 lớp Conv2d+MaxPool2d, in shape sau mỗi lớp.

In [ ]:
# TODO: Bài 3 nâng cao


**Bài 4 (nâng cao):** Thêm Dropout + BatchNorm vào mạng ở Bài 5 (cơ bản), huấn luyện và so sánh loss cuối cùng với bản không có regularization.

In [ ]:
# TODO: Bài 4 nâng cao


**Bài 5 (nâng cao):** Thêm `StepLR` scheduler vào vòng lặp huấn luyện, in learning rate hiện tại (`optimizer.param_groups[0]['lr']`) mỗi 5 epoch.

In [ ]:
# TODO: Bài 5 nâng cao


**Bài 6 (nâng cao):** Huấn luyện một mô hình bất kỳ, lưu bằng `torch.save`, sau đó load lại vào instance mới và dùng để dự đoán trên dữ liệu test giả (nhớ `model.eval()` + `torch.no_grad()`).

In [ ]:
# TODO: Bài 6 nâng cao


---
*Gợi ý: chạy notebook này trên Google Colab (bật GPU runtime) để thấy rõ hơn phần CUDA và CNN.*